In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv('../vehicles_us.csv', parse_dates=['date_posted'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         51525 non-null  int64         
 1   model_year    47906 non-null  float64       
 2   model         51525 non-null  str           
 3   condition     51525 non-null  str           
 4   cylinders     46265 non-null  float64       
 5   fuel          51525 non-null  str           
 6   odometer      43633 non-null  float64       
 7   transmission  51525 non-null  str           
 8   type          51525 non-null  str           
 9   paint_color   42258 non-null  str           
 10  is_4wd        25572 non-null  float64       
 11  date_posted   51525 non-null  datetime64[us]
 12  days_listed   51525 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(2), str(6)
memory usage: 7.2 MB


In [3]:
df.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [4]:
df.describe()

,price,model_year,cylinders,odometer,is_4wd,date_posted,days_listed
count,51525.000000,47906.000000,46265.000000,43633.000000,25572.0,51525,51525.00000
mean,12132.464920,2009.750470,6.125235,115553.461738,1.0,2018-10-25 01:57:46.270742,39.55476
min,1.000000,1908.000000,3.000000,0.000000,1.0,2018-05-01 00:00:00,0.00000
25%,5000.000000,2006.000000,4.000000,70000.000000,1.0,2018-07-29 00:00:00,19.00000
50%,9000.000000,2011.000000,6.000000,113000.000000,1.0,2018-10-25 00:00:00,33.00000
75%,16839.000000,2014.000000,8.000000,155000.000000,1.0,2019-01-21 00:00:00,53.00000
max,375000.000000,2019.000000,12.000000,990000.000000,1.0,2019-04-19 00:00:00,271.00000
std,10040.803015,6.282065,1.660360,65094.611341,0.0,NaN,28.20427


Valores ausentes

In [5]:
missing = (df.isna().mean() * 100).sort_values(ascending=False).round(1)
missing = missing[missing > 0].reset_index()
missing.columns = ['columna', 'porcentaje']

fig = px.bar(missing, x='columna', y='porcentaje', text='porcentaje',
             title='Porcentaje de valores ausentes por columna')
fig.update_layout(yaxis_title='% ausente', xaxis_title=None)
fig.show()

`is_4wd` solo contiene el valor `1`, así que los ausentes significan "no es 4x4" y se rellenan con `0`. Para el resto de columnas se mantienen los ausentes; plotly simplemente los omite en cada gráfico.

In [6]:
df['is_4wd'] = df['is_4wd'].fillna(0).astype(int)
df['model_year'] = df['model_year'].astype('Int64')
df['manufacturer'] = df['model'].str.split().str[0]
df['age'] = df['date_posted'].dt.year - df['model_year']

Distribución de las variables numéricas
Precio

In [7]:
fig = px.histogram(df, x='price', nbins=200, title='Distribución del precio')
fig.update_layout(xaxis_title='Precio (USD)', yaxis_title='Número de anuncios')
fig.show()

La distribución tiene una cola larga a la derecha por unos pocos precios extremos, y hay anuncios con precios sospechosamente bajos. Para los siguientes gráficos se filtran los valores atípicos: precios menores a 500 USD y mayores al percentil 99.

In [8]:
p99 = df['price'].quantile(0.99)
print(f"Anuncios con precio < 500: {(df['price'] < 500).sum()}")
print(f"Percentil 99 del precio: {p99:,.0f}")

df_f = df[(df['price'] >= 500) & (df['price'] <= p99)]
print(f'Filas tras el filtro: {len(df_f)} de {len(df)}')

Anuncios con precio < 500: 1069
Percentil 99 del precio: 43,995
Filas tras el filtro: 49941 de 51525


In [9]:
fig = px.histogram(df_f, x='price', nbins=60, marginal='box',
                   title='Distribución del precio (sin valores atípicos)')
fig.update_layout(xaxis_title='Precio (USD)', yaxis_title='Número de anuncios')
fig.show()

### Kilometraje, año del modelo y días publicados

In [10]:
fig = px.histogram(df_f, x='odometer', nbins=80, marginal='box',
                   title='Distribución del odómetro (millas)')
fig.update_layout(xaxis_title='Millas', yaxis_title='Número de anuncios')
fig.show()

In [11]:
fig = px.histogram(df_f, x='model_year', title='Distribución del año del modelo')
fig.update_layout(xaxis_title='Año del modelo', yaxis_title='Número de anuncios')
fig.show()

In [12]:
fig = px.histogram(df_f, x='days_listed', nbins=60, marginal='box',
                   title='Días que el anuncio estuvo publicado')
fig.update_layout(xaxis_title='Días publicados', yaxis_title='Número de anuncios')
fig.show()

## 4. Variables categóricas

In [13]:
type_counts = df['type'].value_counts().reset_index()
fig = px.bar(type_counts, x='type', y='count', title='Anuncios por tipo de vehículo')
fig.update_layout(xaxis_title=None, yaxis_title='Número de anuncios')
fig.show()

In [14]:
manuf_counts = df['manufacturer'].value_counts().reset_index()
fig = px.bar(manuf_counts, x='manufacturer', y='count', title='Anuncios por fabricante')
fig.update_layout(xaxis_title=None, yaxis_title='Número de anuncios')
fig.show()

In [15]:
condition_order = ['new', 'like new', 'excellent', 'good', 'fair', 'salvage']

fig = px.histogram(df, x='manufacturer', color='type', barmode='stack',
                   title='Tipos de vehículo por fabricante')
fig.update_layout(xaxis_title=None, yaxis_title='Número de anuncios',
                  xaxis={'categoryorder': 'total descending'})
fig.show()

In [16]:
fig = px.histogram(df, x='model_year', color='condition', barmode='stack',
                   category_orders={'condition': condition_order},
                   title='Condición del vehículo según el año del modelo')
fig.update_layout(xaxis_title='Año del modelo', yaxis_title='Número de anuncios')
fig.show()

## 5. Relación del precio con otras variables

In [17]:
fig = px.scatter(df_f, x='odometer', y='price', color='condition', opacity=0.4,
                 category_orders={'condition': condition_order},
                 title='Precio vs. odómetro')
fig.update_layout(xaxis_title='Millas', yaxis_title='Precio (USD)')
fig.show()

In [18]:
fig = px.box(df_f, x='condition', y='price', category_orders={'condition': condition_order},
             title='Precio según la condición')
fig.update_layout(xaxis_title=None, yaxis_title='Precio (USD)')
fig.show()

In [19]:
fig = px.box(df_f, x='type', y='price', title='Precio según el tipo de vehículo')
fig.update_layout(xaxis_title=None, yaxis_title='Precio (USD)',
                  xaxis={'categoryorder': 'median descending'})
fig.show()

In [20]:
price_by_year = (df_f.groupby('model_year', as_index=False)['price'].median()
                 .query('model_year >= 1990'))
fig = px.line(price_by_year, x='model_year', y='price', markers=True,
              title='Precio mediano por año del modelo (desde 1990)')
fig.update_layout(xaxis_title='Año del modelo', yaxis_title='Precio mediano (USD)')
fig.show()

In [21]:
fig = px.box(df_f, x='fuel', y='price', color='transmission',
             title='Precio según combustible y transmisión')
fig.update_layout(xaxis_title=None, yaxis_title='Precio (USD)')
fig.show()

## 6. Correlación entre variables numéricas

In [22]:
corr = df_f[['price', 'model_year', 'age', 'odometer', 'cylinders', 'is_4wd', 'days_listed']].corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Matriz de correlación')
fig.show()

## 7. Conclusiones

- **Calidad de datos:** `is_4wd` (50 % ausente) en realidad codifica "no 4x4"; también faltan valores en `paint_color` (18 %), `odometer` (15 %), `cylinders` (10 %) y `model_year` (7 %). No hay filas duplicadas.
- **Valores atípicos:** ~1 000 anuncios tienen precios menores a 500 USD y hay precios de hasta 375 000 USD. Se filtraron para el análisis (quedan ~49 900 filas).
- **Composición:** SUV, truck y sedan dominan el mercado; Ford y Chevrolet son los fabricantes con más anuncios.
- **Precio:** sube con el año del modelo (r ≈ 0.51) y baja con el kilometraje (r ≈ −0.48). Pickups y trucks son los más caros (mediana ~14 000–15 000 USD) y hatchbacks/sedanes los más baratos (~6 000 USD). Los vehículos diésel tienen una mediana muy superior a los de gasolina.
- **Condición:** el precio mediano baja de forma constante de `new` a `salvage`, como era de esperar.
- **Días publicados:** la mediana es de 33 días y prácticamente no está correlacionada con el precio (r ≈ 0).